# P2 – RGCN and its MLP control A3 (Sơn)

Does message passing over the 6 relations add anything on top of each account's own features?

- **Input (both models):** P + C + T + G, 816 features (P, C, G standardized with train statistics, T raw), the same
  input as P1 Fusion-LR. G stays in, because many accounts have no mention, reply or quote edges (Minh's error analysis).
- **P2 RGCN:** linear input layer (816 → h), then 2 RGCN layers (h → h → 2) with a root (self) weight and one weight
  per relation type. Mean over the neighbours of each relation type (Schlichtkrull et al.). Transductive, full graph.
- **A3 MLP control:** the same network with the neighbour terms removed (only the root weights). Same input, training
  and grid, so the difference is what message passing adds.
- **Training:** Adam, weight decay 5e-4, class-weighted cross-entropy (weights from train), up to 200 epochs, early stop
  on validation PR-AUC (patience 30). Grid: hidden {64, 128}, dropout {0.3, 0.5}, learning rate {0.005, 0.01} with
  seed 0, then the best setting of each model with seeds 0, 1, 2.

Validation only; the test set is not used. Note: model choice, early stopping and threshold all use validation, as for
the B1 MLP in `baselines.ipynb`, so these numbers are a little optimistic for every model alike.

In [1]:
# 1. Setup (Colab or local)
import os, sys
if not os.path.exists("mgtab.py"):
    if "google.colab" in sys.modules:
        if not os.path.exists("sna-content-moderation"):
            !git clone -q https://github.com/ThSonDev/sna-content-moderation.git
        %cd sna-content-moderation
        !pip install -q -r requirements.txt
        !python install_pyg.py
    else:
        %cd ..
sys.path.insert(0, os.getcwd())

/mnt/DevZone/social-networks/project/sna-content-moderation


In [2]:
# 2. Download MGTAB (31 MB) into data/MGTAB, only once
import urllib.request, zipfile
if not os.path.exists("data/MGTAB/features.pt"):
    os.makedirs("data", exist_ok=True)
    url = "https://drive.usercontent.google.com/download?id=1gbWNOoU1JB8RrTu2a5j9KMNVa9wX72Fe&export=download&confirm=t"
    urllib.request.urlretrieve(url, "data/MGTAB.zip")
    zipfile.ZipFile("data/MGTAB.zip").extractall("data")

In [3]:
# 3. Load data and the shared split (never make your own split)
import numpy as np
from mgtab import load, relations, split, best_threshold, evaluate, RELATIONS
X, y, stance, E, T, W = load()
rel = relations(E, T, len(y))   # {"follow": (src, dst), ...}
train, val, test = split(y)     # test stays locked until the final run
print(X.shape, {r: len(s) for r, (s, d) in rel.items()})

(10199, 788) {'follow': 425863, 'mention': 110868, 'reply': 22327, 'quote': 20901, 'url': 263800, 'hashtag': 300000}


## Features and graph

In [4]:
import time
import pandas as pd
import torch
from sklearn.preprocessing import StandardScaler
from features import feature_groups

groups, g_names = feature_groups(X, rel)
Z = {k: StandardScaler().fit(v[train]).transform(v) for k, v in groups.items() if k != "T"}  # train statistics only
x = torch.tensor(np.hstack([Z["P"], Z["C"], groups["T"], Z["G"]]), dtype=torch.float32)
n = len(y)

# 10 relation types: each directed relation and its reverse, URL and hashtag (undirected, stored once) both ways.
types = []
for r, (s, d) in rel.items():
    s, d = torch.as_tensor(np.asarray(s)), torch.as_tensor(np.asarray(d))
    if r in ("url", "hashtag"):
        types.append((r, torch.cat([s, d]), torch.cat([d, s])))
    else:
        types += [(r, s, d), (r + "_rev", d, s)]

def mean_adj(src, dst):
    """A[i, j] = 1 / (number of type-r neighbours of i) for each edge j -> i: A @ h = mean over those neighbours."""
    deg = torch.bincount(dst, minlength=n).clamp(min=1).float()
    return torch.sparse_coo_tensor(torch.stack([dst, src]), 1 / deg[dst], (n, n)).coalesce().to_sparse_csr()

A = [mean_adj(s, d) for _, s, d in types]
print(x.shape, {name: len(s) for name, s, _ in types}, "edges:", sum(len(s) for _, s, _ in types))

/tmp/ipykernel_48655/652649869.py:24: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:848.)
  return torch.sparse_coo_tensor(torch.stack([dst, src]), 1 / deg[dst], (n, n)).coalesce().to_sparse_csr()
/tmp/ipykernel_48655/652649869.py:24: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return torch.sparse_coo_tensor(torch.stack([dst, src]), 1 / deg[dst], (n, n)).coalesce().to_sparse_csr()


torch.Size([10199, 816]) {'follow': 425863, 'follow_rev': 425863, 'mention': 110868, 'mention_rev': 110868, 'reply': 22327, 'reply_rev': 22327, 'quote': 20901, 'quote_rev': 20901, 'url': 527600, 'hashtag': 600000} edges: 2287518


### RGCN layer

PyG's `RGCNConv` was 10× slower here on CPU, and with pyg-lib installed (our setup) it switches on large graphs to a
`segment_matmul` path that takes the mean over *all* incoming edges instead of per relation type (see the
`TODO Re-weight according to edge type degree` in its source). So we use the same formula with one sparse
product per relation, and check it below against `RGCNConv` with that path turned off.

In [5]:
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.nn import RGCNConv

class Layer(nn.Module):
    """h_i' = W_root h_i + b + sum_r mean_{j in N_r(i)} W_r h_j. Without A it is a plain linear layer (MLP)."""
    def __init__(self, i, o, num_rel):
        super().__init__()
        self.root = nn.Linear(i, o)
        self.W = nn.Parameter(torch.empty(num_rel, i, o)) if num_rel else None
        if num_rel:
            nn.init.xavier_uniform_(self.W)

    def forward(self, h, A):
        out = self.root(h)
        return out + sum(torch.sparse.mm(a, h) @ self.W[r] for r, a in enumerate(A)) if self.W is not None else out

class Net(nn.Module):
    def __init__(self, hidden, dropout, num_rel):
        super().__init__()
        self.inp, self.dropout = nn.Linear(x.shape[1], hidden), dropout
        self.l1, self.l2 = Layer(hidden, hidden, num_rel), Layer(hidden, 2, num_rel)

    def forward(self, x, A):
        h = F.dropout(F.relu(self.inp(x)), self.dropout, self.training)
        h = F.dropout(F.relu(self.l1(h, A)), self.dropout, self.training)
        return self.l2(h, A)

# Self-check on 20,000 random edges: same output as RGCNConv (per-relation mean, root weight, bias).
torch.manual_seed(0)
ei, et, hx = torch.randint(0, n, (2, 20000)), torch.randint(0, 10, (20000,)), torch.randn(n, 16)
torch_geometric.backend.use_segment_matmul = False
ref, mine = RGCNConv(16, 8, 10), Layer(16, 8, 10)
with torch.no_grad():
    mine.W.copy_(ref.weight); mine.root.weight.copy_(ref.root.T); mine.root.bias.copy_(ref.bias)
    diff = (mine(hx, [mean_adj(ei[0][et == r], ei[1][et == r]) for r in range(10)]) - ref(hx, ei, et)).abs().max()
assert diff < 1e-4, diff
print("matches RGCNConv, max diff", float(diff))

/mnt/DevZone/social-networks/project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/mnt/DevZone/social-networks/project/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


matches RGCNConv, max diff 4.76837158203125e-07


## Training

In [6]:
from sklearn.metrics import average_precision_score

yt = torch.tensor(y)
tr_idx, va_idx = torch.tensor(train), torch.tensor(val)
counts = np.bincount(y[train])
class_w = torch.tensor(len(train) / (2 * counts), dtype=torch.float32)   # same as class_weight="balanced"

def fit(kind, hidden, dropout, lr, seed, max_epochs=200, patience=30):
    """Returns validation bot probabilities at the best epoch (validation PR-AUC), and that epoch."""
    torch.manual_seed(seed)
    adj = A if kind == "RGCN" else []
    model = Net(hidden, dropout, len(adj))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    best, best_epoch = (-1, None), 0
    for epoch in range(1, max_epochs + 1):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(x, adj)[tr_idx], yt[tr_idx], weight=class_w)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            s = torch.softmax(model(x, adj)[va_idx], 1)[:, 1].numpy()
        ap = average_precision_score(y[val], s)
        if ap > best[0]:
            best, best_epoch = (ap, s), epoch
        elif epoch - best_epoch >= patience:
            break
    return best[1], best_epoch

grid = [(h, d, lr) for h in [64, 128] for d in [0.3, 0.5] for lr in [0.005, 0.01]]
rows = []
for kind in ["MLP", "RGCN"]:
    for h, d, lr in grid:
        t0 = time.time()
        s, ep = fit(kind, h, d, lr, seed=0)
        rows.append({"model": kind, "hidden": h, "dropout": d, "lr": lr, "best_epoch": ep,
                     "pr_auc": average_precision_score(y[val], s), "seconds": round(time.time() - t0)})
        print(rows[-1])
grid_res = pd.DataFrame(rows)
grid_res.to_csv("results/p2_grid.csv", index=False)
grid_res.round(4)

{'model': 'MLP', 'hidden': 64, 'dropout': 0.3, 'lr': 0.005, 'best_epoch': 147, 'pr_auc': 0.9327913918401805, 'seconds': 11}


{'model': 'MLP', 'hidden': 64, 'dropout': 0.3, 'lr': 0.01, 'best_epoch': 135, 'pr_auc': 0.930149080686177, 'seconds': 12}


{'model': 'MLP', 'hidden': 64, 'dropout': 0.5, 'lr': 0.005, 'best_epoch': 78, 'pr_auc': 0.925456878253022, 'seconds': 7}


{'model': 'MLP', 'hidden': 64, 'dropout': 0.5, 'lr': 0.01, 'best_epoch': 99, 'pr_auc': 0.9244128144241897, 'seconds': 8}


{'model': 'MLP', 'hidden': 128, 'dropout': 0.3, 'lr': 0.005, 'best_epoch': 110, 'pr_auc': 0.9278841343148165, 'seconds': 15}


{'model': 'MLP', 'hidden': 128, 'dropout': 0.3, 'lr': 0.01, 'best_epoch': 109, 'pr_auc': 0.9279506294764068, 'seconds': 16}


... (7 outputs left out)

{'model': 'RGCN', 'hidden': 128, 'dropout': 0.3, 'lr': 0.01, 'best_epoch': 144, 'pr_auc': 0.9257578698194082, 'seconds': 86}


{'model': 'RGCN', 'hidden': 128, 'dropout': 0.5, 'lr': 0.005, 'best_epoch': 142, 'pr_auc': 0.926172399171069, 'seconds': 89}


{'model': 'RGCN', 'hidden': 128, 'dropout': 0.5, 'lr': 0.01, 'best_epoch': 87, 'pr_auc': 0.8966236933520503, 'seconds': 54}


,model,hidden,dropout,lr,best_epoch,pr_auc,seconds
0,MLP,64,0.3,0.005,147,0.9328,11
1,MLP,64,0.3,0.010,135,0.9301,12
2,MLP,64,0.5,0.005,78,0.9255,7
3,MLP,64,0.5,0.010,99,0.9244,8
4,MLP,128,0.3,0.005,110,0.9279,15
5,MLP,128,0.3,0.010,109,0.9280,16
6,MLP,128,0.5,0.005,84,0.9275,19
7,MLP,128,0.5,0.010,178,0.9282,24
8,RGCN,64,0.3,0.005,150,0.9281,66
9,RGCN,64,0.3,0.010,180,0.9181,77


## Best setting, 3 seeds

In [7]:
best_cfg = grid_res.loc[grid_res.groupby("model")["pr_auc"].idxmax()].set_index("model")
names = {"MLP": "A3 MLP control (P+C+T+G)", "RGCN": "P2 RGCN (P+C+T+G)"}
rows, seed_scores = [], {}
for kind in ["MLP", "RGCN"]:
    c = best_cfg.loc[kind]
    for seed in [0, 1, 2]:
        s, ep = fit(kind, int(c.hidden), float(c.dropout), float(c.lr), seed)
        seed_scores[(kind, seed)] = s
        rows.append({"model": names[kind], "seed": seed, "hidden": int(c.hidden), "dropout": c.dropout, "lr": c.lr,
                     "best_epoch": ep, **evaluate(y[val], s, best_threshold(y[val], s))})
per_seed = pd.DataFrame(rows)
metrics = ["pr_auc", "roc_auc", "macro_f1", "accuracy", "p@100", "p@200"]
summary = per_seed.groupby("model", sort=False)[metrics].agg(["mean", "std"])
per_seed.to_csv("results/p2.csv", index=False)
display(per_seed.round(4))
summary.round(4)

,model,seed,hidden,dropout,lr,best_epoch,pr_auc,roc_auc,macro_f1,accuracy,p@100,p@200,threshold
0,A3 MLP control (P+C+T+G),0,64,0.3,0.005,147,0.9328,0.9784,0.9176,0.9333,0.97,0.945,0.6721
1,A3 MLP control (P+C+T+G),1,64,0.3,0.005,181,0.9324,0.9780,0.9188,0.9346,0.96,0.950,0.7436
2,A3 MLP control (P+C+T+G),2,64,0.3,0.005,143,0.9306,0.9772,0.9149,0.9320,0.97,0.950,0.6788
3,P2 RGCN (P+C+T+G),0,128,0.3,0.005,173,0.9373,0.9789,0.9186,0.9359,0.98,0.960,0.5232
4,P2 RGCN (P+C+T+G),1,128,0.3,0.005,190,0.9262,0.9748,0.9142,0.9307,0.98,0.955,0.5773
5,P2 RGCN (P+C+T+G),2,128,0.3,0.005,99,0.9224,0.9722,0.8969,0.9176,0.98,0.955,0.6191


pr_auc         roc_auc         macro_f1          \
                            mean     std    mean     std     mean     std   
model                                                                       
A3 MLP control (P+C+T+G)  0.9319  0.0012  0.9779  0.0006   0.9171  0.0020   
P2 RGCN (P+C+T+G)         0.9287  0.0077  0.9753  0.0034   0.9099  0.0115   

                         accuracy           p@100           p@200          
                             mean     std    mean     std    mean     std  
model                                                                      
A3 MLP control (P+C+T+G)   0.9333  0.0013  0.9667  0.0058  0.9483  0.0029  
P2 RGCN (P+C+T+G)          0.9281  0.0094  0.9800  0.0000  0.9567  0.0029

## Comparisons

Paired bootstrap of the PR-AUC change (1,000 resamples of the validation accounts), using each model's scores averaged
over the 3 seeds. P1 Fusion-LR is refit here with its chosen C = 0.1 from `p1.ipynb`.

In [8]:
from sklearn.linear_model import LogisticRegression

def lr_scores(F, C):
    return LogisticRegression(C=C, class_weight="balanced", max_iter=5000).fit(F[train], y[train]).predict_proba(F[val])[:, 1]

content = np.hstack([Z["P"], Z["C"], groups["T"]])
avg = {kind: np.mean([seed_scores[(kind, s)] for s in [0, 1, 2]], 0) for kind in ["MLP", "RGCN"]}
avg["P1 Fusion-LR"] = lr_scores(x.numpy(), 0.1)
avg["content LR"] = lr_scores(content, 10)   # C = 10 as chosen for the fair baseline

rng = np.random.default_rng(42)
yv = y[val]
boots = [rng.choice(len(val), len(val)) for _ in range(1000)]
def delta(a, b):
    d = [average_precision_score(yv[i], avg[a][i]) - average_precision_score(yv[i], avg[b][i]) for i in boots]
    return {"comparison": f"{a} - {b}", "delta_pr_auc": average_precision_score(yv, avg[a]) - average_precision_score(yv, avg[b]),
            **dict(zip(["ci_low", "ci_high"], np.percentile(d, [2.5, 97.5])))}
pd.DataFrame([delta("RGCN", "MLP"), delta("RGCN", "P1 Fusion-LR"), delta("MLP", "P1 Fusion-LR")]).round(4)

,comparison,delta_pr_auc,ci_low,ci_high
0,RGCN - MLP,-0.0006,-0.0165,0.0154
1,RGCN - P1 Fusion-LR,0.0177,-0.0006,0.0370
2,MLP - P1 Fusion-LR,0.0182,0.0071,0.0297


### Minh's check set: quiet humans

Humans that P1 Fusion-LR gets wrong and content-only LR gets right (Minh's `hurt` group; each LR with its own
threshold). Minh found the graph features push these low-activity humans toward bot. Does the RGCN, which also sees
their neighbours, call fewer of them bots? Counts per seed, each seed with its own validation threshold.

In [9]:
pred = {k: avg[k] >= best_threshold(yv, avg[k]) for k in ["P1 Fusion-LR", "content LR"]}
quiet = (yv == 0) & pred["P1 Fusion-LR"] & ~pred["content LR"]
check = {"quiet humans": int(quiet.sum()), "called bot by P1 Fusion-LR": int(quiet.sum())}
for kind in ["MLP", "RGCN"]:
    for seed in [0, 1, 2]:
        s = seed_scores[(kind, seed)]
        check[f"called bot by {kind}, seed {seed}"] = int((s >= best_threshold(yv, s))[quiet].sum())
pd.Series(check)

quiet humans                  17
called bot by P1 Fusion-LR    17
called bot by MLP, seed 0      9
called bot by MLP, seed 1      8
called bot by MLP, seed 2      6
called bot by RGCN, seed 0     6
called bot by RGCN, seed 1     9
called bot by RGCN, seed 2    10
dtype: int64

## Findings (validation, one split)

1. **Message passing adds nothing measurable.** Over 3 seeds, P2 RGCN reaches PR-AUC 0.929 ± 0.008 and the A3 MLP
   control (same input and network, no neighbours) 0.932 ± 0.001. On seed-averaged scores RGCN − MLP is −0.001
   (95% CI −0.017 to +0.015). Once each account's own graph features (G) are in the input, the neighbours' features
   add no signal we can detect.
2. **The RGCN is less stable.** Its seed-0 grid run (0.937) was the best single run, but seeds 1 and 2 gave 0.926 and
   0.922; the MLP stays within 0.931–0.933. Choosing the grid winner on seed 0 favours a lucky run.
3. **A non-linear model beats P1 Fusion-LR on the same 816 features:** MLP − P1 = +0.018 PR-AUC (CI +0.007 to
   +0.030); macro-F1 0.917 vs 0.906. RGCN − P1 = +0.018, but its CI touches 0 (−0.001 to +0.037).
4. **Quiet humans:** of the 17 humans that P1 Fusion-LR wrongly calls bots and content-only LR gets right (Minh found 18
   with his own threshold run; the group shifts by one account), the MLP still calls 6–9 bots and the RGCN 6–10.
   Neighbours do not help with them more than a non-linear model on their own features does.

Caveats: one split; grid, early stopping and threshold all use validation (same for every model here). Only P + C + T + G
input was tried; the RGCN relation ablation (A5) and the isolated-relation flags are not done yet.